In [4]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
!pip install numpy pandas matplotlib seaborn

import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")   # 학습 중 경고 메시지를 잠시 숨깁니다.

# 재현성: 같은 난수를 항상 같게 만들어 결과가 매번 동일하도록 고정합니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록 운영체제별로 분기)
system = platform.system()
if system == "Darwin":          # macOS
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":       # Windows
    plt.rcParams["font.family"] = "Malgun Gothic"
else:                            # Linux 등
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False   # 마이너스 부호 깨짐 방지
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.4.6
pandas: 3.0.3


In [5]:
# ─────────────────────────────────────────────
# 모두마켓 데이터 생성 — 이 셀 하나로 오늘 쓸 데이터가 모두 준비됩니다.
# (실제 현장처럼 '오염'을 일부러 심어 둡니다: 결측·이상치·표기 혼재·날짜 포맷 혼재·중복)
# ─────────────────────────────────────────────
np.random.seed(42)

# 1) 고객(customers)
n_customers = 300
customers = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(4)}" for i in range(1, n_customers + 1)],
    "age": np.random.normal(35, 9, n_customers).round().astype(int),
    "gender": np.random.choice(["M", "F"], n_customers),
    "region": np.random.choice(["서울", "경기", "부산", "인천", "대구"], n_customers),
    "membership": np.random.choice(["basic", "premium", "vip"], n_customers, p=[0.6, 0.3, 0.1]),
})
# 오염 심기: 나이 이상치, 결측, 지역/멤버십 표기 혼재
customers.loc[5, "age"] = 999          # 입력 실수로 보이는 이상치
customers.loc[10, "age"] = -3          # 음수 나이(불가능한 값)
customers.loc[[20, 21, 22], "gender"] = np.nan
customers.loc[30, "region"] = " 서울 "  # 앞뒤 공백
customers.loc[31, "region"] = "Seoul"   # 영문 표기 혼재
customers.loc[40, "membership"] = "VIP"  # 대소문자 혼재

# 2) 상품(products)
categories = ["패션", "뷰티", "식품", "가전", "도서"]
n_products = 40
products = pd.DataFrame({
    "product_id": [f"P{str(i).zfill(3)}" for i in range(1, n_products + 1)],
    "category": np.random.choice(categories, n_products),
    "price": np.random.choice([9900, 19900, 29900, 49900, 89900, 129900], n_products),
})

# 3) 주문(orders)
n_orders = 2000
order_customer = np.random.choice(customers["customer_id"], n_orders)
order_product = np.random.choice(products["product_id"], n_orders)
price_map = products.set_index("product_id")["price"]
quantity = np.random.choice([1, 1, 1, 2, 2, 3], n_orders)
amount = price_map.loc[order_product].values * quantity

orders = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n_orders + 1)],
    "customer_id": order_customer,
    "product_id": order_product,
    "quantity": quantity,
    "amount": amount.astype(float),
    "channel": np.random.choice(["web", "app", "app ", "APP"], n_orders, p=[0.45, 0.45, 0.05, 0.05]),
})

# 날짜 포맷 혼재(문자열로 저장 — 일부러 통일하지 않음)
base_dates = pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 120, n_orders), unit="D")
date_strings = []
for i, d in enumerate(base_dates):
    if i % 3 == 0:
        date_strings.append(d.strftime("%Y-%m-%d"))
    elif i % 3 == 1:
        date_strings.append(d.strftime("%Y/%m/%d"))
    else:
        date_strings.append(d.strftime("%Y%m%d"))
orders["order_date"] = date_strings

# 오염 심기: 금액 결측, 수량 이상치, 중복 행
orders.loc[np.random.choice(n_orders, 35, replace=False), "amount"] = np.nan
orders.loc[7, "quantity"] = 100        # 비정상적으로 큰 주문 수량
orders = pd.concat([orders, orders.iloc[[0, 1]]], ignore_index=True)  # 중복 주문 2건

print("모두마켓 데이터 생성 완료")
print("customers:", customers.shape, "| products:", products.shape, "| orders:", orders.shape)

모두마켓 데이터 생성 완료
customers: (300, 5) | products: (40, 3) | orders: (2002, 7)


In [6]:
# 데이터가 어떻게 생겼는지 맨 위 몇 줄만 살짝 들여다봅니다.
print("=== orders (주문) ===")
display(orders.head())
print("\n=== customers (고객) ===")
display(customers.head())

=== orders (주문) ===


,order_id,customer_id,product_id,quantity,amount,channel,order_date
0,O00001,C0040,P017,1,19900.0,web,2025-03-25
1,O00002,C0224,P022,1,89900.0,app,2025/04/24
2,O00003,C0115,P034,1,49900.0,app,20250405
3,O00004,C0186,P029,1,89900.0,web,2025-01-04
4,O00005,C0056,P004,3,149700.0,web,2025/04/28



=== customers (고객) ===


,customer_id,age,gender,region,membership
0,C0001,39,M,경기,premium
1,C0002,34,F,부산,basic
2,C0003,41,F,서울,premium
3,C0004,49,M,대구,vip
4,C0005,33,M,서울,basic


In [7]:
# 예제: 기본 자료형 4종을 확인합니다. type()은 값의 자료형을 알려줍니다.
age = 23                # 정수(int)
avg_amount = 200409.25  # 실수(float)
region = "김포"         # 문자열(str)
is_premium = True       # 불리언(bool)

for value in [age, avg_amount, region, is_premium]:
    print(f"{str(value):>8}  →  {type(value).__name__}")

      23  →  int
200409.25  →  float
      김포  →  str
    True  →  bool


In [8]:
# 예제: 컬렉션 4종을 모두마켓 맥락으로 만들어 봅니다.
order_amounts = [19900, 29900, 9900, 19900]      # 리스트: 주문 금액들 (순서·중복 O)
store_location = (37.5665, 126.9780)             # 튜플: 본사 좌표 (수정 X)
regions_seen = {"서울", "경기", "부산", "서울"}    # 집합: 중복이 자동으로 사라짐
customer = {"id": "C0001", "age": 35, "region": "서울"}  # 딕셔너리: 이름표로 접근

print("리스트(주문 금액):", order_amounts, "| 합계:", sum(order_amounts))
print("튜플(본사 좌표):", store_location)
print("집합(등장한 지역):", regions_seen, "  ← '서울' 중복이 사라짐")
print("딕셔너리(고객):", customer["region"], "에 사는", customer["age"], "세 고객")

리스트(주문 금액): [19900, 29900, 9900, 19900] | 합계: 79600
튜플(본사 좌표): (37.5665, 126.978)
집합(등장한 지역): {'서울', '경기', '부산'}   ← '서울' 중복이 사라짐
딕셔너리(고객): 서울 에 사는 35 세 고객


In [ ]:
# 스스로 해보자! (1)
# 아래 주석(#)을 지우고 빈칸(___)을 채운 뒤 실행해보세요.

my_customer = {
     "id": "C0099",
     "age": 23,         # 빈칸을 채우세요
     "region": "김포",
     "is_premium": True,
 }
print(my_customer["age"])   # 2) 나이만 출력


# 3)고객을 100명을 담는다면, 딕셔너리들을 담은 리스트가 좋을 것 같다. 지금처럼 나이만 가지고 오는 경우에는, 나이만 딕셔너리 하나 형태로 담을 수 있겠지만, 리스트로 만들어두면 다른 컬럼에 대해서도 언제든 유사한 작업을 계속 할 수 있기 때문

23


In [13]:
#스스로 해보자! (2)  

def mean_amount(amounts):
    if len(amounts) == 0:       #리스트니까 len 사용
        return 0
    return sum(amounts) / len(amounts)

print(mean_amount([19900, 29900, 9900]))  # 19900.0


19900.0


In [11]:
# 스스로 해보자! (3)
import numpy as np
scores = np.array([1,2,3,4,5,6,4,4,123,413,5,6,54,567,654])
print("평균: ",scores.mean())
print("표준편차 : ",scores.std().round(1))
above = scores > scores.mean()
print(above)
print(above.sum())

평균:  123.4
표준편차 :  217.5
[False False False False False False False False False  True False False
 False  True  True]
3
